# AgriVision v9 — Model penyaring: **daun tomat** atau **bukan daun tomat**

Model diagnosis (10 kelas) selalu menjawab salah satu penyakit tomat, bahkan untuk daun cabai atau foto tembok. Notebook ini melatih **model kecil kedua** (MobileNetV2, keluaran biner) yang dipasang **sebelum** diagnosis:

```
Foto → penyaring: peluang "daun tomat" ≥ ambang? ── tidak → ditolak "Foto bukan daun tomat"
                                                  └ ya → model diagnosis 10 kelas
```

**Prioritas (keputusan user):** daun tomat asli **tidak boleh** ditolak. Ambang dipilih pada data validasi supaya **≥ 98% daun tomat diterima di setiap sumber** (lab, PlantDoc, Tomato-Village); baru setelah itu sebanyak mungkin daun lain ditolak.

**Data (semua diunduh otomatis, tidak ada upload dataset):**

| Kelompok | Sumber | Lisensi |
|---|---|---|
| Tomat | PlantVillage (10 kelas), PlantDoc (tomat), Tomato-Village (**8 kelas**, termasuk Leaf Miner, Spotted Wilt, kekurangan Mg/N/K — semuanya tetap daun tomat) | CC BY-SA 3.0 / CC BY-SA 4.0 / riset, wajib sitasi |
| Cabai | Roboflow Universe (3 proyek, butuh API key gratis — opsional), Kaggle `dhenyd/chili-plant-disease` (butuh `kaggle.json` — opsional), Hugging Face `Project-AgML/COLD_chili_leaf_disease_classification` (tanpa kunci) | CC BY 4.0 (Roboflow/COLD) |
| Paprika, kentang, apel, anggur, jagung, stroberi, kedelai, labu, ceri, persik, jeruk, blueberry, raspberry | PlantVillage + PlantDoc | CC BY-SA |
| Terong | Hugging Face `Project-AgML/eggplant_disease_classification` | CC BY 4.0 |
| Singkong | TensorFlow Datasets `cassava` (iCassava 2019) | lihat katalog TFDS |
| Bukan daun: benda | Hugging Face `johnowhitaker/imagenette2-320` (Imagenette) | Apache 2.0 (Imagenette asli) |
| Bukan daun: tekstur (tembok, kain, retakan, dll.) | Hugging Face `tanganke/dtd` (Describable Textures) | riset |

Pembagian **70% latih : 10% validasi : 20% uji** per sumber per jenis (PlantVillage per daun), foto hampir kembar dibuang sebelum dibagi.

**Menjalankan:** Runtime → **T4 GPU** → Runtime → **Run all**. Di **Step 1** ada dua isian opsional (Roboflow API key, `kaggle.json`); tekan **Enter / Cancel** untuk melewati. Perkiraan ±1–1,5 jam. Model dan hasil disimpan ke Google Drive (`MyDrive/AgriVision_v9`); kalau sesi putus, **Run all** lagi — training dilewati bila model sudah ada. Sel terakhir mengunduh `hasil_v9.zip` (model TFLite, ambang, tabel hasil, contoh foto yang salah).

In [ ]:
# Cek GPU aktif — kalau kosong/error: Runtime > Change runtime type > T4 GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Step 0 — Install, import, dan pengaturan

In [ ]:
!pip install -q datasets scikit-learn seaborn roboflow kaggle
import os, re, io, json, time, glob, shutil, random, hashlib, subprocess, zipfile
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import tensorflow as tf
from PIL import Image

try:
    display
except NameError:
    display = print

SEED = 42
RATIO = {"train": 0.70, "val": 0.10, "test": 0.20}
IMG_SIZE = (224, 224)
BATCH = 32
EPOCHS_HEAD, EPOCHS_FT = 5, 10
NEAR_DUP_THR = 3
TARGET_TERIMA_TOMAT = 0.98      # minimal 98% daun tomat diterima di SETIAP sumber (pada validasi)

# Batas jumlah foto per jenis supaya seimbang dan training tetap cepat
CAP_PV_TOMAT_PER_KELAS = 600
CAP_PV_LAIN_PER_TANAMAN = 400
CAP_PV_LAIN_KHUSUS = {"Paprika": 1200, "Kentang": 1200}   # paling mirip tomat -> porsi lebih
CAP_TERONG, CAP_SINGKONG, CAP_BENDA, CAP_TEKSTUR = 1000, 600, 1000, 600

POOL_DIR, DATA_DIR, OUT_DIR = "/content/pool_v9", "/content/data_v9", "/content/out_v9"
USE_DRIVE, DRIVE_DIR = True, "/content/drive/MyDrive/AgriVision_v9"
SAVE_DIR = "/content/simpan_v9"
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        SAVE_DIR = DRIVE_DIR
    except Exception as e:
        print("Google Drive tidak dipakai:", e)
for d in (SAVE_DIR, OUT_DIR):
    os.makedirs(d, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print("TensorFlow:", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"), "| simpan ke:", SAVE_DIR)

## Step 1 — Kunci API (opsional, untuk foto cabai tambahan)

- **Roboflow:** daftar gratis di https://app.roboflow.com → Settings → API Keys → salin *Private API Key*, lalu tempel di kotak yang muncul. Kosongkan + Enter untuk melewati.
- **Kaggle:** pilih file `kaggle.json` milikmu. Klik *Cancel upload* untuk melewati.

Kunci hanya dipakai di sesi Colab ini dan **tidak disimpan** ke Drive maupun hasil.

In [ ]:
# (sel unduhan data)
import getpass
ROBOFLOW_API_KEY = ""
HAS_KAGGLE = False
try:
    ROBOFLOW_API_KEY = getpass.getpass("Roboflow Private API Key (kosongkan + Enter untuk lewati): ").strip()
except Exception as e:
    print("Roboflow dilewati:", e)
try:
    from google.colab import files
    print("Pilih kaggle.json (atau klik Cancel upload untuk lewati)")
    up = files.upload()
    if "kaggle.json" in up:
        os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
        with open(os.path.expanduser("~/.kaggle/kaggle.json"), "wb") as f:
            f.write(up["kaggle.json"])
        os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
        os.remove("kaggle.json") if os.path.exists("kaggle.json") else None
        HAS_KAGGLE = True
except Exception as e:
    print("Kaggle dilewati:", e)
print("Roboflow:", "ya" if ROBOFLOW_API_KEY else "dilewati", "| Kaggle:", "ya" if HAS_KAGGLE else "dilewati")

## Step 2 — Fungsi bantu

Setiap foto dicatat dengan: `kind` (**tomat** / **bukan**), `jenis` (mis. Cabai, Kentang, Tomat), `src` (sumber dataset), `cond` (kondisi/kelas asal, untuk laporan), `group` (foto satu grup selalu masuk bagian yang sama).

In [ ]:
POOL = []
shutil.rmtree(POOL_DIR, ignore_errors=True)
os.makedirs(POOL_DIR, exist_ok=True)

def dhash(img, size=8):
    g = img.convert("L").resize((size + 1, size), Image.LANCZOS)
    a = np.asarray(g, dtype=np.int16)
    bits = (a[:, 1:] > a[:, :-1]).flatten()
    return int("".join("1" if b else "0" for b in bits), 2)

def popcount64(a):
    a = np.ascontiguousarray(np.atleast_1d(a).astype(np.uint64))
    return np.unpackbits(a.view(np.uint8).reshape(-1, 8), axis=1).sum(axis=1)

def norm(s):
    return re.sub(r"[^a-z0-9]", "", str(s).lower())

def add_to_pool(img, kind, jenis, src, cond, group=None):
    try:
        img = img.convert("RGB")
        if min(img.size) < 32:
            return
        i = len(POOL)
        path = f"{POOL_DIR}/{i:06d}.png"
        img.resize(IMG_SIZE, Image.BILINEAR).save(path, compress_level=1)
        POOL.append({"id": i, "kind": kind, "jenis": jenis, "src": src, "cond": str(cond),
                     "group": group or f"{src}|{i}", "h": dhash(img), "path": path})
    except Exception as e:
        print("  foto dilewati (rusak):", e)

def add_images_from_dir(root, kind, jenis, src, cap=None, cond_from_parent=True):
    paths = sorted(p for p in glob.glob(f"{root}/**/*", recursive=True)
                   if p.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp")))
    if cap and len(paths) > cap:
        paths = sorted(random.Random(f"{SEED}|{src}").sample(paths, cap))
    for p in paths:
        with Image.open(p) as im:
            add_to_pool(im, kind, jenis, src, os.path.basename(os.path.dirname(p)) if cond_from_parent else "")
    return len(paths)

def ringkas(title):
    df = pd.DataFrame([(r["kind"], r["jenis"], r["src"]) for r in POOL], columns=["kind", "jenis", "src"])
    print(title); display(df.value_counts().rename("jumlah").reset_index().sort_values(["kind", "jenis", "src"], ascending=[False, True, True]).reset_index(drop=True))

# Sitasi/sumber setiap dataset yang benar-benar terpakai (ditulis ke hasil)
SUMBER = {}

## Step 3 — PlantVillage: tomat (10 kelas) + 13 tanaman lain

Foto dikelompokkan per **daun** (`leaf_id`). Jumlah per kelas/tanaman dibatasi (lihat Step 0) dengan memilih daun secara acak (seed tetap).

In [ ]:
# (sel unduhan data)
from huggingface_hub import hf_hub_download

REPO = "mohanty/PlantVillage"
zip_path = hf_hub_download(REPO, "data.zip", repo_type="dataset")   # +-2,2 GB
leaf_map = json.load(open(hf_hub_download(REPO, "leaf_grouping/leaf-map.json", repo_type="dataset"), encoding="utf-8"))
zf = zipfile.ZipFile(zip_path)

def leaf_id_for(filename, class_name):
    name = filename.replace("_final_masked", "")
    if "___" in name:
        name = name.split("___")[-1]
    low = name.lower()
    if "copy" in low:
        name = name[: low.index("copy")]
    for ext in (".jpg", ".png"):
        if name.lower().endswith(ext):
            name = name[: -len(ext)]
    ident = name.strip().lower()
    sugg = leaf_map.get(ident)
    if isinstance(sugg, str):
        sugg = [sugg]
    if sugg:
        if len(sugg) == 1:
            return str(sugg[0])
        for sg in sugg:
            if class_name in str(sg):
                return str(sg)
    return f"fallback_{ident}"

PV_CROP_ID = {"Apple": "Apel", "Blueberry": "Blueberry", "Cherry_(including_sour)": "Ceri", "Corn_(maize)": "Jagung",
              "Grape": "Anggur", "Orange": "Jeruk", "Peach": "Persik", "Pepper,_bell": "Paprika", "Potato": "Kentang",
              "Raspberry": "Raspberry", "Soybean": "Kedelai", "Squash": "Labu", "Strawberry": "Stroberi", "Tomato": "Tomat"}

by_bucket = defaultdict(lambda: defaultdict(list))   # bucket -> leaf -> [member]
for n in zf.namelist():
    if "raw/color/" not in n or n.endswith("/"):
        continue
    parts = n[n.index("raw/color/"):].split("/")
    if len(parts) < 4:
        continue
    folder, fname = parts[2], parts[-1]
    crop = folder.split("___")[0]
    if crop not in PV_CROP_ID:
        print("  tanaman tidak dikenal, dilewati:", folder); continue
    bucket = folder if crop == "Tomato" else crop        # tomat: per kelas penyakit; lain: per tanaman
    by_bucket[bucket][leaf_id_for(fname, folder)].append((n, folder))

t0 = time.time()
for bucket in sorted(by_bucket):
    leaves = by_bucket[bucket]
    crop = bucket.split("___")[0]
    jenis = PV_CROP_ID[crop]
    cap = CAP_PV_TOMAT_PER_KELAS if crop == "Tomato" else CAP_PV_LAIN_KHUSUS.get(jenis, CAP_PV_LAIN_PER_TANAMAN)
    keys = sorted(leaves); random.Random(f"{SEED}|pv|{bucket}").shuffle(keys)
    taken = 0
    for k in keys:
        if taken >= cap:
            break
        for member, folder in sorted(leaves[k]):
            with zf.open(member) as fh:
                img = Image.open(fh).convert("RGB")
            add_to_pool(img, "tomat" if crop == "Tomato" else "bukan", jenis, "PlantVillage",
                        folder.split("___")[-1], f"pv|{folder}|{k}")
            taken += 1
    print(f"  {bucket:45s} -> {taken} foto ({time.time() - t0:.0f} dtk)")
SUMBER["PlantVillage"] = "Hughes & Salathé (2015), huggingface.co/datasets/mohanty/PlantVillage, CC BY-SA 3.0"
ringkas("Setelah PlantVillage:")

## Step 4 — PlantDoc: tomat + tanaman lain (foto non-lab)

In [ ]:
# (sel unduhan data)
from datasets import load_dataset

pdoc = load_dataset("Project-AgML/plant_doc_classification", split="train")
names = pdoc.features["label"].names
PD_JENIS = [("tomato", "Tomat"), ("bell_pepper", "Paprika"), ("potato", "Kentang"), ("apple", "Apel"), ("grape", "Anggur"),
            ("corn", "Jagung"), ("strawberry", "Stroberi"), ("soyabean", "Kedelai"), ("squash", "Labu"), ("cherry", "Ceri"),
            ("peach", "Persik"), ("blueberry", "Blueberry"), ("raspberry", "Raspberry")]

def pd_jenis(label_name):
    low = label_name.lower().replace(" ", "_")
    for key, jenis in PD_JENIS:
        if low.startswith(key):
            return jenis
    return None

skipped = Counter()
for ex in pdoc:
    nm = names[ex["label"]]
    jenis = pd_jenis(nm)
    if jenis is None:
        skipped[nm] += 1; continue
    add_to_pool(ex["image"], "tomat" if jenis == "Tomat" else "bukan", jenis, "PlantDoc", nm)
if skipped:
    print("  kelas PlantDoc tidak dipetakan:", dict(skipped))
SUMBER["PlantDoc"] = "Singh dkk. (2020), huggingface.co/datasets/Project-AgML/plant_doc_classification, CC BY-SA 4.0"
ringkas("Setelah PlantDoc:")

## Step 5 — Tomato-Village: **semua 8 kelas** (semuanya daun tomat)

Untuk penyaring, kelas yang tidak ada di model diagnosis (Leaf Miner, Spotted Wilt Virus, kekurangan Mg/N/K) **tetap dipakai sebagai tomat**, supaya daun tomat dengan kondisi apa pun tidak ditolak.

In [ ]:
# (sel unduhan data)
cmds = """
set -e
rm -rf /content/ext && mkdir -p /content/ext && cd /content/ext
git clone -q --filter=blob:none --no-checkout --depth 1 https://github.com/mamta-joshi-gehlot/Tomato-Village.git tv
cd tv
git sparse-checkout init --cone
git sparse-checkout set "Variant-a(Multiclass Classification)"
git checkout -q
"""
subprocess.run(["bash", "-c", cmds], check=True)
n = add_images_from_dir("/content/ext/tv/Variant-a(Multiclass Classification)", "tomat", "Tomat", "Tomato-Village")
print("Tomato-Village:", n, "foto")
SUMBER["Tomato-Village"] = "Gehlot, Saxena & Gandhi (2023), github.com/mamta-joshi-gehlot/Tomato-Village (riset, wajib sitasi)"

## Step 6 — Cabai (paling mirip tomat): Roboflow + Kaggle + Hugging Face

Setiap sumber dicoba; kalau gagal (kunci tidak diisi, proyek berubah), notebook **lanjut** dengan sumber lain dan mencatatnya.

In [ ]:
# (sel unduhan data)
ROBOFLOW_PROJECTS = [   # (workspace, project) — proyek publik di universe.roboflow.com
    ("chili-leaves-disease-classification", "chili-leaves-disease-classification"),
    ("tanaman-cabai", "chili-plant-disease-v4"),
    ("syiah-kuala-university-8vtha", "chili-leaf-disease"),
]
if ROBOFLOW_API_KEY:
    from roboflow import Roboflow
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    for ws, pid in ROBOFLOW_PROJECTS:
        try:
            project = rf.workspace(ws).project(pid)
            fmt = "folder" if getattr(project, "type", "") == "classification" else "yolov8"
            ver = None
            for vn in range(15, 0, -1):
                try:
                    ver = project.version(vn); break
                except Exception:
                    continue
            assert ver is not None, "tidak ada versi yang bisa diakses"
            loc = f"/content/ext/rf_{pid}"
            ver.download(fmt, location=loc)
            n = add_images_from_dir(loc, "bukan", "Cabai", f"Roboflow:{pid}")
            print(f"  Roboflow {ws}/{pid} v{vn} ({fmt}): {n} foto")
            SUMBER[f"Roboflow:{pid}"] = f"universe.roboflow.com/{ws}/{pid} (versi {vn})"
        except Exception as e:
            print(f"  Roboflow {ws}/{pid} GAGAL, dilewati: {e}")
else:
    print("Roboflow dilewati (tanpa API key)")

if HAS_KAGGLE:
    try:
        subprocess.run(["kaggle", "datasets", "download", "-d", "dhenyd/chili-plant-disease", "-p", "/content/ext/kg_chili", "--unzip"], check=True)
        n = add_images_from_dir("/content/ext/kg_chili", "bukan", "Cabai", "Kaggle:dhenyd/chili-plant-disease")
        print("  Kaggle dhenyd/chili-plant-disease:", n, "foto")
        SUMBER["Kaggle:dhenyd/chili-plant-disease"] = "kaggle.com/datasets/dhenyd/chili-plant-disease"
    except Exception as e:
        print("  Kaggle GAGAL, dilewati:", e)
else:
    print("Kaggle dilewati (tanpa kaggle.json)")

try:
    cold = load_dataset("Project-AgML/COLD_chili_leaf_disease_classification", "raw", split="train")
    cn = cold.features["label"].names
    for ex in cold:
        add_to_pool(ex["image"], "bukan", "Cabai", "HF:COLD_chili", cn[ex["label"]])
    print("  Hugging Face COLD chili:", len(cold), "foto")
    SUMBER["HF:COLD_chili"] = "Aishwarya & Reddy (2024), huggingface.co/datasets/Project-AgML/COLD_chili_leaf_disease_classification, CC BY 4.0"
except Exception as e:
    print("  COLD chili GAGAL, dilewati:", e)

## Step 7 — Terong dan singkong (tanaman lokal)

In [ ]:
# (sel unduhan data)
try:   # raw = foto resolusi tinggi (+-2,7 GB total) -> dibaca streaming, diambil sebagian saja
    egg = load_dataset("Project-AgML/eggplant_disease_classification", "raw", split="train", streaming=True)
    egg = egg.shuffle(seed=SEED, buffer_size=200)
    en = None; k = 0
    for ex in egg:
        if k >= CAP_TERONG:
            break
        add_to_pool(ex["image"], "bukan", "Terong", "HF:eggplant", ex["label"]); k += 1
    print("  Terong:", k, "foto")
    SUMBER["HF:eggplant"] = "huggingface.co/datasets/Project-AgML/eggplant_disease_classification (Data in Brief), CC BY 4.0"
except Exception as e:
    print("  Terong GAGAL, dilewati:", e)

try:
    import tensorflow_datasets as tfds
    cas = tfds.load("cassava", split="train", shuffle_files=False)
    k = 0
    for ex in cas.shuffle(2000, seed=SEED).take(CAP_SINGKONG):
        add_to_pool(Image.fromarray(ex["image"].numpy()), "bukan", "Singkong", "TFDS:cassava", int(ex["label"])); k += 1
    print("  Singkong:", k, "foto")
    SUMBER["TFDS:cassava"] = "Mwebaze dkk. (2019) iCassava, tensorflow.org/datasets/catalog/cassava"
except Exception as e:
    print("  Singkong GAGAL, dilewati:", e)

## Step 8 — Bukan daun: benda sehari-hari dan tekstur (tembok, kain, retakan, dll.)

In [ ]:
# (sel unduhan data)
for repo, jenis, cap in [("johnowhitaker/imagenette2-320", "Benda umum", CAP_BENDA), ("tanganke/dtd", "Tekstur", CAP_TEKSTUR)]:
    try:
        ds = load_dataset(repo, split="train").shuffle(seed=SEED).select(range(cap))
        lab = ds.features["label"].names if hasattr(ds.features["label"], "names") else None
        for ex in ds:
            if jenis == "Tekstur" and lab and lab[ex["label"]] in ("veined",):   # tekstur 'veined' bisa berisi urat daun -> buang
                continue
            add_to_pool(ex["image"], "bukan", jenis, f"HF:{repo.split('/')[-1]}", lab[ex["label"]] if lab else ex["label"])
        print(f"  {jenis}: {cap} foto")
        SUMBER[f"HF:{repo.split('/')[-1]}"] = f"huggingface.co/datasets/{repo}"
    except Exception as e:
        print(f"  {jenis} GAGAL, dilewati:", e)
ringkas("SEMUA sumber (sebelum dibersihkan):")

## Step 9 — Buang foto yang hampir kembar

Untuk semua sumber selain PlantVillage (PlantVillage dijaga per daun): kembar di dalam sumber disisakan satu; kembar **lintas sumber** dibuang dari sumber yang belakangan; kalau kembar tapi **beda kelompok** (tomat vs bukan), keduanya dibuang.

In [ ]:
def dedup():
    order = sorted(POOL, key=lambda r: (r["src"] != "PlantVillage", r["id"]))   # PlantVillage jadi acuan
    kept_h = np.zeros(len(order), dtype=np.uint64); kept = []; drop = set(); why = Counter()
    for r in order:
        if kept and r["src"] != "PlantVillage":
            hit = np.where(popcount64(kept_h[: len(kept)] ^ np.uint64(r["h"])) <= NEAR_DUP_THR)[0]
            if len(hit):
                drop.add(r["id"])
                for j in hit:
                    if kept[j]["kind"] != r["kind"]:
                        drop.add(kept[j]["id"]); why["kembar tapi beda kelompok (keduanya dibuang)"] += 1
                why[f"kembar ({r['jenis']}, {r['src']})"] += 1
                continue
        kept_h[len(kept)] = np.uint64(r["h"]); kept.append(r)
    return drop, why

DROP, why = dedup()
for k, v in sorted(why.items()):
    print(f"  dibuang — {k}: {v}")
for r in POOL:
    if r["id"] in DROP:
        os.remove(r["path"])
POOL = [r for r in POOL if r["id"] not in DROP]
print("Total foto setelah dibersihkan:", len(POOL))

## Step 10 — Bagi 70/10/20 per sumber per jenis

In [ ]:
SPLITS = ["train", "val", "test"]

def split_one(records, rng):
    groups = defaultdict(list)
    for r in records:
        groups[r["group"]].append(r)
    keys = sorted(groups); rng.shuffle(keys)
    n = len(records)
    t_test = max(1, round(RATIO["test"] * n)) if n >= 3 else 0
    t_val = max(1, round(RATIO["val"] * n)) if n >= 10 else 0
    out = {sp: [] for sp in SPLITS}
    for k in keys:
        g = groups[k]
        if len(out["test"]) < t_test:
            out["test"] += g
        elif len(out["val"]) < t_val:
            out["val"] += g
        else:
            out["train"] += g
    return out

shutil.rmtree(DATA_DIR, ignore_errors=True)
buckets = defaultdict(list)
for r in POOL:
    buckets[(r["kind"], r["jenis"], r["src"])].append(r)
for key in sorted(buckets):
    for sp, rs in split_one(buckets[key], random.Random(f"{SEED}|{key}")).items():
        for r in rs:
            r["split"] = sp
            d = f"{DATA_DIR}/{sp}/{r['kind']}"
            os.makedirs(d, exist_ok=True)
            dst = f"{d}/{r['id']:06d}.png"
            shutil.move(r["path"], dst); r["path"] = dst

for src in {r["src"] for r in POOL}:
    seen = defaultdict(set)
    for r in POOL:
        if r["src"] == src:
            seen[r["group"]].add(r["split"])
    assert not [g for g, s in seen.items() if len(s) > 1], f"{src}: ada grup di dua bagian"
print("OK — tidak ada daun/foto yang sama di dua bagian")

manifest = pd.DataFrame([{k: (f"{r[k]:016x}" if k == "h" else r[k]) for k in ("kind", "jenis", "src", "cond", "split", "group", "h")} for r in POOL])
manifest = manifest.sort_values(list(manifest.columns)).reset_index(drop=True)
FINGERPRINT = hashlib.sha1(manifest.to_csv(index=False).encode()).hexdigest()
manifest.to_csv(f"{OUT_DIR}/manifest_v9.csv", index=False)
print("Sidik jari data:", FINGERPRINT[:12])

## Step 11 — Tabel data (latih / validasi / uji)

In [ ]:
def tri(rs):
    c = Counter(r["split"] for r in rs)
    return f"{c['train']} / {c['val']} / {c['test']}"

rows = []
for (kind, jenis, src), rs in sorted(buckets.items(), key=lambda x: (x[0][0] != "tomat", x[0][1], x[0][2])):
    rs = [r for r in POOL if (r["kind"], r["jenis"], r["src"]) == (kind, jenis, src)]
    if rs:
        rows.append({"Kelompok": "TOMAT" if kind == "tomat" else "bukan tomat", "Jenis": jenis, "Sumber": src,
                     "Latih / validasi / uji": tri(rs), "Total": len(rs)})
for kind in ["tomat", "bukan"]:
    rs = [r for r in POOL if r["kind"] == kind]
    rows.append({"Kelompok": "TOTAL " + ("TOMAT" if kind == "tomat" else "bukan tomat"), "Jenis": "", "Sumber": "",
                 "Latih / validasi / uji": tri(rs), "Total": len(rs)})
TABEL_DATA = pd.DataFrame(rows)
display(TABEL_DATA)
TABEL_DATA.to_csv(f"{OUT_DIR}/tabel_1_data.csv", index=False)

## Step 12 — Latih MobileNetV2 biner

Resep sama dengan model diagnosis (tahap 1 kepala, tahap 2 buka 60 lapisan atas, BatchNorm beku, augmentasi ringan), tetapi keluaran **satu angka**: peluang "daun tomat". Class weight menyeimbangkan tomat vs bukan. Early stopping memantau AUC validasi. Model disimpan ke Drive; kalau sudah ada (data sama), training dilewati.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
REC = {sp: [r for r in POOL if r["split"] == sp] for sp in SPLITS}

def _load(path, label):
    img = tf.io.decode_png(tf.io.read_file(path), channels=3)
    return tf.ensure_shape(img, IMG_SIZE + (3,)), label

def make_ds(recs, train):
    if train:
        recs = [recs[i] for i in np.random.RandomState(SEED).permutation(len(recs))]
    paths = [r["path"] for r in recs]
    labels = np.array([1.0 if r["kind"] == "tomat" else 0.0 for r in recs], dtype=np.float32)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels)).map(_load, num_parallel_calls=AUTOTUNE).cache()
    if train:
        ds = ds.shuffle(4000, seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(BATCH).map(lambda x, y: (tf.cast(x, tf.float32), y), num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE), labels

def make_augment():
    return tf.keras.Sequential([
        tf.keras.layers.RandomFlip("horizontal_and_vertical"),
        tf.keras.layers.RandomRotation(0.1),
        tf.keras.layers.RandomZoom(0.15),
        tf.keras.layers.RandomTranslation(0.05, 0.05),
        tf.keras.layers.RandomBrightness(0.15, value_range=(0, 255)),
        tf.keras.layers.RandomContrast(0.15),
    ], name="augment")

def build_gate():
    inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
    x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)   # input mentah 0-255
    base = tf.keras.applications.MobileNetV2(input_shape=IMG_SIZE + (3,), include_top=False, weights="imagenet")
    base.trainable = False
    x = base(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(1, activation="sigmoid", name="p_tomat")(x)
    return tf.keras.Model(inputs, outputs, name="gate_tomat"), base

def compile_gate(m, lr):
    m.compile(optimizer=tf.keras.optimizers.Adam(lr), loss="binary_crossentropy",
              metrics=["accuracy", tf.keras.metrics.AUC(name="auc")])

MODEL_PATH = f"{SAVE_DIR}/gate_{FINGERPRINT[:12]}.keras"
train_ds, y_train = make_ds(REC["train"], True)
val_ds, y_val = make_ds(REC["val"], False)
test_ds, y_test = make_ds(REC["test"], False)
print(f"latih {len(y_train)} (tomat {int(y_train.sum())}) | validasi {len(y_val)} | uji {len(y_test)}")

if os.path.exists(MODEL_PATH):
    gate = tf.keras.models.load_model(MODEL_PATH)
    print("Model sudah ada di Drive — training dilewati:", MODEL_PATH)
    TRAIN_INFO = json.load(open(MODEL_PATH + ".info.json"))
else:
    tf.keras.utils.set_random_seed(SEED)
    gate, base = build_gate()
    model = tf.keras.Sequential([tf.keras.Input(shape=IMG_SIZE + (3,)), make_augment(), gate], name="train_wrapper")
    n_pos = float(y_train.sum()); n_neg = float(len(y_train) - n_pos)
    cw = {0: len(y_train) / (2 * n_neg), 1: len(y_train) / (2 * n_pos)}
    stop = lambda p: [tf.keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=p, restore_best_weights=True)]
    t0 = time.time()
    compile_gate(model, 1e-3)
    h1 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD, class_weight=cw, callbacks=stop(2), verbose=2)
    base.trainable = True
    for layer in base.layers[:-60]:
        layer.trainable = False
    for layer in base.layers:
        if isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = False
    compile_gate(model, 1e-4)
    h2 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FT, class_weight=cw, callbacks=stop(3), verbose=2)
    TRAIN_INFO = {"epochs": [len(h1.history["loss"]), len(h2.history["loss"])],
                  "best_val_auc": float(max(h1.history["val_auc"] + h2.history["val_auc"])),
                  "minutes": round((time.time() - t0) / 60, 1), "class_weight": cw}
    gate.save(MODEL_PATH)
    json.dump(TRAIN_INFO, open(MODEL_PATH + ".info.json", "w"))
    del model
print(TRAIN_INFO)

## Step 13 — Pilih ambang (di data **validasi**)

Ambang = angka tertinggi yang masih membuat **setiap sumber tomat** (PlantVillage, PlantDoc, Tomato-Village) diterima ≥ 98% di validasi. Makin tinggi ambang, makin banyak daun lain yang ditolak — tapi makin berisiko menolak daun tomat. Tabel di bawah menunjukkan tarik-ulurnya.

In [ ]:
p_val = gate.predict(val_ds, verbose=0).ravel()
p_test = gate.predict(test_ds, verbose=0).ravel()
for r, p in zip(REC["val"], p_val):
    r["p"] = float(p)
for r, p in zip(REC["test"], p_test):
    r["p"] = float(p)

TOMAT_SRC = sorted({r["src"] for r in POOL if r["kind"] == "tomat"})

def terima_per_sumber(recs, t):
    return {s: float(np.mean([r["p"] >= t for r in recs if r["kind"] == "tomat" and r["src"] == s])) for s in TOMAT_SRC
            if any(r["kind"] == "tomat" and r["src"] == s for r in recs)}

def tolak_bukan(recs, t):
    return float(np.mean([r["p"] < t for r in recs if r["kind"] == "bukan"]))

grid = np.round(np.arange(0.02, 0.99, 0.01), 2)
ok = [t for t in grid if min(terima_per_sumber(REC["val"], t).values()) >= TARGET_TERIMA_TOMAT]
THRESHOLD = float(max(ok)) if ok else 0.5
print(f"Ambang terpilih: {THRESHOLD:.2f}" + ("" if ok else "  (PERINGATAN: target tidak tercapai, memakai 0,50)"))

rows = []
for t in sorted(set([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, THRESHOLD])):
    tv = terima_per_sumber(REC["val"], t)
    rows.append({"Ambang": t, **{f"Tomat diterima — {s}": f"{v * 100:.1f}%" for s, v in tv.items()},
                 "Bukan tomat ditolak": f"{tolak_bukan(REC['val'], t) * 100:.1f}%", "Dipilih": "◀" if t == THRESHOLD else ""})
TABEL_AMBANG = pd.DataFrame(rows)
print("Tabel ambang (data validasi):"); display(TABEL_AMBANG)
TABEL_AMBANG.to_csv(f"{OUT_DIR}/tabel_2_ambang_validasi.csv", index=False)

## Step 14 — Hasil di data **uji** (belum pernah dilihat model)

- **Tabel 3:** per jenis/sumber — untuk tomat: % **diterima**; untuk bukan tomat: % **ditolak**.
- **Tabel 4:** tomat per kondisi/kelas (memastikan daun tomat sakit parah pun tetap diterima).
- Gambar contoh salah: tomat yang ditolak dan bukan-tomat yang lolos.

In [ ]:
import matplotlib.pyplot as plt

T = THRESHOLD
rows = []
for (kind, jenis, src) in sorted({(r["kind"], r["jenis"], r["src"]) for r in REC["test"]}, key=lambda k: (k[0] != "tomat", k[1], k[2])):
    rs = [r for r in REC["test"] if (r["kind"], r["jenis"], r["src"]) == (kind, jenis, src)]
    ok_rate = np.mean([(r["p"] >= T) if kind == "tomat" else (r["p"] < T) for r in rs])
    rows.append({"Kelompok": "TOMAT" if kind == "tomat" else "bukan tomat", "Jenis": jenis, "Sumber": src, "n uji": len(rs),
                 "Benar (%)": round(ok_rate * 100, 1), "Arti": "diterima" if kind == "tomat" else "ditolak"})
for kind in ["tomat", "bukan"]:
    rs = [r for r in REC["test"] if r["kind"] == kind]
    ok_rate = np.mean([(r["p"] >= T) if kind == "tomat" else (r["p"] < T) for r in rs])
    rows.append({"Kelompok": "TOTAL " + ("TOMAT" if kind == "tomat" else "bukan tomat"), "Jenis": "", "Sumber": "", "n uji": len(rs),
                 "Benar (%)": round(ok_rate * 100, 1), "Arti": "diterima" if kind == "tomat" else "ditolak"})
TABEL_UJI = pd.DataFrame(rows)
print(f"Tabel 3. Hasil uji per jenis (ambang {T:.2f})"); display(TABEL_UJI)

rows = []
for (src, cond) in sorted({(r["src"], r["cond"]) for r in REC["test"] if r["kind"] == "tomat"}):
    rs = [r for r in REC["test"] if r["kind"] == "tomat" and r["src"] == src and r["cond"] == cond]
    rows.append({"Sumber": src, "Kondisi/kelas": cond, "n uji": len(rs), "Diterima (%)": round(np.mean([r["p"] >= T for r in rs]) * 100, 1)})
TABEL_TOMAT = pd.DataFrame(rows)
print("\nTabel 4. Daun tomat per kondisi — % diterima"); display(TABEL_TOMAT)

from sklearn.metrics import roc_auc_score
y = np.array([r["kind"] == "tomat" for r in REC["test"]], dtype=int); p = np.array([r["p"] for r in REC["test"]])
METRIK = {"threshold": T, "auc_uji": float(roc_auc_score(y, p)),
          "tomat_diterima_uji": float(np.mean(p[y == 1] >= T)), "bukan_ditolak_uji": float(np.mean(p[y == 0] < T)),
          "akurasi_uji": float(np.mean((p >= T) == (y == 1))), "n_uji": int(len(y)), "train": TRAIN_INFO, "fingerprint": FINGERPRINT}
print("\nRingkasan:", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in METRIK.items() if k not in ("train",)})
TABEL_UJI.to_csv(f"{OUT_DIR}/tabel_3_uji_per_jenis.csv", index=False)
TABEL_TOMAT.to_csv(f"{OUT_DIR}/tabel_4_tomat_per_kondisi.csv", index=False)

def galeri(recs, title, fname):
    recs = recs[:16]
    if not recs:
        print(title, ": tidak ada"); return
    cols = 8; rws = (len(recs) + cols - 1) // cols
    fig, axes = plt.subplots(rws, cols, figsize=(2.4 * cols, 2.8 * rws))
    for ax in np.atleast_1d(axes).ravel():
        ax.axis("off")
    for ax, r in zip(np.atleast_1d(axes).ravel(), recs):
        ax.imshow(Image.open(r["path"])); ax.set_title(f"{r['jenis']}\n{r['src'][:18]}\np={r['p']:.2f}", fontsize=7)
    fig.suptitle(title); plt.tight_layout(); plt.savefig(f"{OUT_DIR}/{fname}", dpi=110); plt.show()

galeri(sorted([r for r in REC["test"] if r["kind"] == "tomat" and r["p"] < T], key=lambda r: r["p"]),
       "Daun TOMAT yang salah DITOLAK (paling yakin salah dulu)", "salah_tomat_ditolak.png")
galeri(sorted([r for r in REC["test"] if r["kind"] == "bukan" and r["p"] >= T], key=lambda r: -r["p"]),
       "BUKAN tomat yang salah LOLOS", "salah_bukan_lolos.png")

## Step 15 — Ekspor TFLite (float16) dan verifikasi

Format sama dengan model diagnosis: input mentah 0–255 (`preprocess_input` di dalam graf), keluaran 1 angka = peluang daun tomat. Dicek: keputusan TFLite (terima/tolak) harus ≥ 99% sama dengan Keras.

In [ ]:
os.makedirs("/content/gate", exist_ok=True)
gate.export("/content/gate/saved_model")
conv = tf.lite.TFLiteConverter.from_saved_model("/content/gate/saved_model")
conv.optimizations = [tf.lite.Optimize.DEFAULT]
conv.target_spec.supported_types = [tf.float16]
tfl = conv.convert()
TFLITE_PATH = f"{OUT_DIR}/tomato_gate.tflite"
open(TFLITE_PATH, "wb").write(tfl)
print(f"TFLite: {len(tfl) / 1e6:.2f} MB")

try:
    interp = tf.lite.Interpreter(model_path=TFLITE_PATH)
except AttributeError:          # TF >= 2.20: tf.lite.Interpreter dipindah ke LiteRT
    from ai_edge_litert.interpreter import Interpreter
    interp = Interpreter(model_path=TFLITE_PATH)
interp.allocate_tensors()
i_d, o_d = interp.get_input_details()[0], interp.get_output_details()[0]
p_tfl = []
for r in REC["test"]:
    x = np.asarray(Image.open(r["path"]).convert("RGB"), dtype=np.float32)[None]
    interp.set_tensor(i_d["index"], x); interp.invoke()
    p_tfl.append(float(interp.get_tensor(o_d["index"]).ravel()[0]))
p_tfl = np.array(p_tfl)
agree = float(np.mean((p_tfl >= T) == (p >= T)))
print(f"Keputusan TFLite sama dengan Keras: {agree * 100:.2f}% | selisih peluang maks {np.abs(p_tfl - p).max():.4f}")
assert agree >= 0.99, "TFLite menyimpang dari Keras — jangan dipasang"

GATE_CONFIG = {"model_version": "gate-v1", "img_size": list(IMG_SIZE), "threshold": round(T, 2),
               "output": "p_tomat (sigmoid, 0-1)", "input": "RGB 0-255 float32, preprocessing di dalam graf",
               "source": "notebook v9 (agrivision_gate_v9_penyaring_tomat.ipynb)", "fingerprint": FINGERPRINT[:12]}
json.dump(GATE_CONFIG, open(f"{OUT_DIR}/gate_config.json", "w"), indent=2)
METRIK["tflite_agreement"] = agree; METRIK["tflite_mb"] = len(tfl) / 1e6; METRIK["sumber"] = SUMBER
json.dump(METRIK, open(f"{OUT_DIR}/metrik_v9.json", "w"), indent=2, default=float)
print(json.dumps(GATE_CONFIG, indent=2))

## Step 16 — Simpan ke Drive dan unduh hasil

In [ ]:
for f in os.listdir(OUT_DIR):
    shutil.copy(f"{OUT_DIR}/{f}", f"{SAVE_DIR}/{f}")
shutil.make_archive("/content/hasil_v9", "zip", OUT_DIR)
print("Isi hasil_v9.zip:", sorted(os.listdir(OUT_DIR)))
try:
    from google.colab import files
    files.download("/content/hasil_v9.zip")
except Exception as e:
    print("Unduh manual dari panel Files (/content/hasil_v9.zip):", e)